In [0]:
#install openpyxl
%pip install openpyxl


In [0]:
#The last line of the output says you may need to restart Python so the notebook can use the new package.
dbutils.library.restartPython()

In [0]:
#sheet-name check cell
import pandas as pd

CATALOG = "capstone_supply_chain"
excel_path = f"/Volumes/{CATALOG}/bronze/landing_excel/inventory_and_quality.xlsx"

print(pd.ExcelFile(excel_path).sheet_names)

In [0]:
#Excel  →  table name in Bronze
#Load the Excel file into Bronze
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit

CATALOG = "capstone_supply_chain"
excel_path = f"/Volumes/{CATALOG}/bronze/landing_excel/inventory_and_quality.xlsx"

# sheet name in Excel  →  table name in Bronze
sheet_table_map = {
    "Inventory_Master": "raw_inventory_master",
    "Quality_Inspections": "raw_quality_inspections",
    "Plant_Information": "raw_plant_info",
}

for sheet_name, table_name in sheet_table_map.items():
    # 1. Read one sheet with pandas
    pdf = pd.read_excel(excel_path, sheet_name=sheet_name)

    # 2. Delta tables don't allow spaces or special characters in column names
    pdf.columns = [str(c).strip().replace(" ", "_") for c in pdf.columns]

    # 3. Convert to a Spark DataFrame and add audit columns
    df = (spark.createDataFrame(pdf)
        .withColumn("_ingested_at", current_timestamp())
        .withColumn("_source_file", lit(excel_path))
        .withColumn("_source_sheet", lit(sheet_name))
        .withColumn("_file_format", lit("xlsx"))
    )

    # 4. Save as a Bronze table
    df.write.mode("overwrite").saveAsTable(f"{CATALOG}.bronze.{table_name}")
    print(f"✅ '{sheet_name}' → {CATALOG}.bronze.{table_name} ({df.count()} rows)")